# Split by near copies, and label noise from box crops

This notebook tests two things on **real photos** (COCO128) with **real DINOv2-small embeddings**:

1. **`split`**: a re-split that keeps near copies of a photo on one side. COCO128 is checked for near copies first; then a few
   re-encoded copies are planted in the validation split, the leakage check finds them, `split --near-dup` writes a new dataset
   version, and the leakage check runs again on it.
2. **`label_noise` with box crops**: the measurement of notebook 04 (seeded planted class mistakes, five seeds) is repeated with
   the new default rule, which embeds every box crop, and with the whole-image rule it replaced, side by side.

Every number on this page is computed by the cell above it; the text written by hand contains no results.

**About the data and its licence.** The photos come from COCO (train2017) through the COCO128 sample published by Ultralytics.
The COCO annotations are licensed CC BY 4.0, and each image keeps the licence of its original Flickr owner. The notebook
downloads the sample at run time into a cache folder outside the repository and never copies, commits or redistributes an image.

### Set-up

The next cell prepares a clean work folder and defines `datacopilot(...)`, which runs a real DataCopilot command exactly as a
terminal would and prints the command, its output and its exit code.

In [1]:
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
import types
import urllib.request
import warnings
import zipfile
from pathlib import Path

# Where downloads are cached (outside the repository) and where this notebook works.
# A private folder that only this user can enter (not a shared, predictable path in the temp folder);
# set DATACOPILOT_NOTEBOOK_DATA to keep the download between runs.
DATA = Path(
    os.environ.get("DATACOPILOT_NOTEBOOK_DATA") or tempfile.mkdtemp(prefix="datacopilot-nb-")
).resolve()
WORK = DATA / "work-label-noise"
shutil.rmtree(WORK, ignore_errors=True)  # start clean, so the notebook can be re-run from the top
WORK.mkdir(parents=True)
os.chdir(WORK)  # everything below uses paths relative to this folder

# Printing helper: hide the folders of this machine behind short placeholders.
_HIDE = [
    (str(p), name)
    for p, name in [
        (WORK, "<work>"),
        (WORK.resolve(), "<work>"),
        (DATA, "<data>"),
        (DATA.resolve(), "<data>"),
        (Path(tempfile.gettempdir()), "<tmp>"),
        (Path(tempfile.gettempdir()).resolve(), "<tmp>"),
        (Path.home(), "~"),
    ]
]
_HIDE.sort(key=lambda pair: -len(pair[0]))


def tidy(text):
    for real, placeholder in _HIDE:
        text = text.replace(real, placeholder)
    return text


class Done(types.SimpleNamespace):
    """What a command returned (shows nothing when it ends a cell)."""

    def _ipython_display_(self):
        pass


def datacopilot(*args, env=None, timed=False):
    """Run one `datacopilot` command like a terminal: show the command, what it prints and its exit code."""
    start = time.perf_counter()
    done = subprocess.run(  # noqa: S603  (our own command, no shell)
        [
            sys.executable,
            "-I",
            "-m",
            "datacopilot",
            *map(str, args),
        ],  # -I: no current-folder or PYTHON* imports
        capture_output=True,
        text=True,
        timeout=3600,
        env={**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", **(env or {})},
    )
    seconds = time.perf_counter() - start
    done.messages = [
        tidy(line) for line in done.stderr.splitlines() if line.strip() and not line.startswith("{")
    ]
    print("$ datacopilot " + " ".join(shlex.quote(str(a)) for a in args))
    for line in done.stdout.splitlines():
        print(tidy(line)[:240])
    for line in done.messages:
        print("[message]", line[:400])
    print(f"[exit code {done.returncode}" + (f", {seconds:.1f} s]" if timed else "]"))
    return Done(
        returncode=done.returncode,
        stdout=done.stdout,
        stderr=done.stderr,
        messages=done.messages,
        seconds=seconds,
    )


def latest_report(name="report.json"):
    """The report the last audit wrote next to us (JSON version)."""
    return json.loads(Path(name).read_text())


os.environ["DATACOPILOT_LOG_LEVEL"] = "ERROR"  # keep the technical JSON log lines out of this page
os.environ["TQDM_DISABLE"] = os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = (
    "1"  # no progress bars on this page
)
warnings.filterwarnings("ignore")
print("cache folder :", tidy(str(DATA)))
print("work folder  :", tidy(str(WORK)))
print("python       :", sys.version.split()[0])

cache folder : <data>
work folder  : <work>
python       : 3.13.9


**What this shows.** Every file this notebook creates lives in the work folder; paths printed below are relative to it.

## Download the real photos, verified

The same pinned download as the other notebooks: the SHA-256 is compared **before** anything is unpacked, and every entry of the
archive is checked. The model is pinned by DataCopilot's own registry (one commit, verified weights, no pickle file).

In [2]:
# Pinned fingerprint of the one file we download (see the note above this cell for where it came from).
COCO128_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
COCO128_SHA256 = "61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e"
COCO128_BYTES = 6_983_030
MAX_DOWNLOAD_BYTES = 50_000_000  # refuse anything bigger than this, whatever the server says
MAX_FILES, MAX_UNPACKED_BYTES, MAX_RATIO = 1_000, 100_000_000, 20


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def fetch_coco128():
    """Download COCO128 (HTTPS only, bounded, atomic); return its path once the pinned checksum matches."""
    if not COCO128_URL.startswith("https://"):
        raise RuntimeError("downloads must use HTTPS")
    DATA.mkdir(parents=True, exist_ok=True)
    target = DATA / "coco128.zip"
    if target.exists() and sha256_of(target) != COCO128_SHA256:
        target.unlink()  # never reuse a cached file whose checksum is wrong
    if not target.exists():
        partial = target.with_suffix(".part")
        try:
            with (
                urllib.request.urlopen(COCO128_URL, timeout=60) as reply,
                open(partial, "wb") as out,
            ):  # noqa: S310
                if int(reply.headers.get("Content-Length") or 0) > MAX_DOWNLOAD_BYTES:
                    raise RuntimeError("the server announced a file that is too big")
                received = 0
                for block in iter(lambda: reply.read(1 << 20), b""):
                    received += len(block)
                    if received > MAX_DOWNLOAD_BYTES:
                        raise RuntimeError("download is larger than the allowed maximum")
                    out.write(block)
        except OSError as problem:
            partial.unlink(missing_ok=True)
            raise RuntimeError(
                "Could not download COCO128 (" + type(problem).__name__ + "). "
                "These notebooks need the network once to fetch about 7 MB from "
                + COCO128_URL
                + ". "
                "Connect and run this cell again, or download that file yourself and save it as "
                "'coco128.zip' in the cache folder (set DATACOPILOT_NOTEBOOK_DATA to choose where it is)."
            ) from None
        except RuntimeError:
            partial.unlink(missing_ok=True)
            raise
        actual = sha256_of(partial)
        if actual != COCO128_SHA256:
            partial.unlink()
            raise RuntimeError(
                "COCO128 checksum mismatch (got "
                + actual
                + "); the file was deleted and nothing was unpacked"
            )
        partial.replace(target)  # atomic: a half-written file never carries the final name
    return target


def unpack_safely(archive_path, destination):
    """Unpack a zip we do not trust: check every entry first, then write into a fresh folder."""
    destination = Path(destination)
    if destination.exists():
        raise RuntimeError("the destination must be a fresh folder")
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        if len(members) > MAX_FILES or sum(m.file_size for m in members) > MAX_UNPACKED_BYTES:
            raise RuntimeError("archive has too many files or unpacks to too much data")
        for member in members:
            name = member.filename
            is_link = ((member.external_attr >> 16) & 0o170000) == 0o120000
            if (
                name.startswith(("/", chr(92)))
                or ".." in Path(name).parts
                or ":" in name
                or is_link
            ):
                raise RuntimeError("unsafe entry in the archive: " + name)
            if member.compress_size and member.file_size / member.compress_size > MAX_RATIO:
                raise RuntimeError("suspiciously high compression ratio for " + name)
        for member in members:  # every entry passed; now write them one by one
            path = destination / member.filename
            if member.is_dir():
                path.mkdir(parents=True, exist_ok=True)
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source_file, open(path, "wb") as out:
                shutil.copyfileobj(source_file, out)
    return destination


zip_path = fetch_coco128()
print("file    :", zip_path.name, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
print("sha256  :", sha256_of(zip_path))
print(
    "checksum matches the pinned value: True (a mismatch stops the notebook before anything is unpacked)"
)

raw = unpack_safely(zip_path, "raw")
print("images  :", len(list((raw / "coco128/images/train2017").glob("*.jpg"))))
print("labels  :", len(list((raw / "coco128/labels/train2017").glob("*.txt"))))

file    : coco128.zip (7.0 MB)
sha256  : 61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e
checksum matches the pinned value: True (a mismatch stops the notebook before anything is unpacked)
images  : 128
labels  : 128


**What this shows.** The archive matched the pinned checksum before it was opened.

## 1. Two copies of the dataset

`coco128` is the folder of notebook 04 (every fifth photo, sorted by name, is "val"). `coco128_copies` is the same folder plus
five **planted near copies**: five train photos, chosen by a seeded draw, saved again as JPEG at a lower quality and slightly
shrunk, and put in the validation split with the same labels. That is what a leak usually looks like: the same scene, other bytes.

In [3]:
import numpy as np
from PIL import Image as PILImage

# COCO128 is one folder of 128 photos with no list of class names and no validation split, so we
# add both: the 80 official COCO class names, and every fifth photo (sorted by name) becomes "val".
NAMES = [
    n.replace("_", " ")
    for n in "person bicycle car motorcycle airplane bus train truck boat traffic_light fire_hydrant stop_sign parking_meter bench bird cat dog horse sheep cow elephant bear zebra giraffe backpack umbrella handbag tie suitcase frisbee skis snowboard sports_ball kite baseball_bat baseball_glove skateboard surfboard tennis_racket bottle wine_glass cup fork knife spoon bowl banana apple sandwich orange broccoli carrot hot_dog pizza donut cake chair couch potted_plant bed dining_table toilet tv laptop mouse remote keyboard cell_phone microwave oven toaster sink refrigerator book clock vase scissors teddy_bear hair_drier toothbrush".split()
]
if len(NAMES) != 80:
    raise ValueError("expected the 80 COCO class names")


def build_yolo_layout(target="coco128"):
    """Copy the raw COCO128 files into the folder layout DataCopilot's YOLO reader expects."""
    target = Path(target)
    source = raw / "coco128"
    photos = sorted((source / "images/train2017").glob("*.jpg"))
    for number, photo in enumerate(photos):
        split = "val" if number % 5 == 4 else "train"
        (target / "images" / split).mkdir(parents=True, exist_ok=True)
        (target / "labels" / split).mkdir(parents=True, exist_ok=True)
        shutil.copy(photo, target / "images" / split / photo.name)
        label = source / "labels/train2017" / (photo.stem + ".txt")
        if label.exists():
            shutil.copy(label, target / "labels" / split / label.name)
    names = ", ".join(json.dumps(n) for n in NAMES)
    (target / "data.yaml").write_text(
        f"path: .\ntrain: images/train\nval: images/val\nnc: 80\nnames: [{names}]\n"
    )
    return target


build_yolo_layout("coco128")
build_yolo_layout("coco128_copies")

COPY_SEED = 7
train_photos = sorted(Path("coco128_copies/images/train").glob("*.jpg"))
originals = [
    train_photos[i]
    for i in sorted(np.random.default_rng(COPY_SEED).choice(len(train_photos), 5, replace=False))
]
planted_copies = {}
for photo in originals:
    with PILImage.open(photo) as img:
        smaller = img.convert("RGB").resize((round(img.width * 0.95), round(img.height * 0.95)))
    target = Path("coco128_copies/images/val") / ("copy_" + photo.name)
    smaller.save(target, quality=70)
    shutil.copy(
        Path("coco128_copies/labels/train") / (photo.stem + ".txt"),
        Path("coco128_copies/labels/val") / ("copy_" + photo.stem + ".txt"),
    )
    planted_copies[photo.name] = target.name
print(
    "train photos:",
    len(list(Path("coco128/images/train").glob("*.jpg"))),
    "| val photos:",
    len(list(Path("coco128/images/val").glob("*.jpg"))),
)
print(
    "with copies : train",
    len(list(Path("coco128_copies/images/train").glob("*.jpg"))),
    "| val",
    len(list(Path("coco128_copies/images/val").glob("*.jpg"))),
)
for original, copy in planted_copies.items():
    print(f"  planted: images/val/{copy} is a re-encoded copy of images/train/{original}")

train photos: 103 | val photos: 25
with copies : train 103 | val 30
  planted: images/val/copy_000000000404.jpg is a re-encoded copy of images/train/000000000404.jpg
  planted: images/val/copy_000000000431.jpg is a re-encoded copy of images/train/000000000431.jpg
  planted: images/val/copy_000000000474.jpg is a re-encoded copy of images/train/000000000474.jpg
  planted: images/val/copy_000000000595.jpg is a re-encoded copy of images/train/000000000595.jpg
  planted: images/val/copy_000000000605.jpg is a re-encoded copy of images/train/000000000605.jpg


**What this shows.** The two folders and the five planted copies, each named after the train photo it copies.

In [4]:
from collections import Counter

import numpy as np

datacopilot("ingest", "coco128", "--format", "yolo", "-o", "coco128.dcp")
datacopilot("ingest", "coco128_copies", "--format", "yolo", "-o", "copies.dcp")
embed = datacopilot("embed", "coco128.dcp", "--embedder", "dinov2_small", timed=True)
datacopilot("embed", "copies.dcp", "--embedder", "dinov2_small", timed=True)
EMBED_SECONDS = embed.seconds

$ datacopilot ingest coco128 --format yolo -o coco128.dcp
d9105ffb854e7dbeccf255a4848bd692
[exit code 0]


$ datacopilot ingest coco128_copies --format yolo -o copies.dcp
9466d5324b8d9667ae63ba1865d0e982
[exit code 0]


$ datacopilot embed coco128.dcp --embedder dinov2_small
embedded=128 total=128 cached=0 missing=0 model=dinov2_small@ed25f3a31f01 run_id=7bb2e61a-6193-47e4-9580-40c76916c21d
[exit code 0, 5.5 s]


$ datacopilot embed copies.dcp --embedder dinov2_small
embedded=133 total=133 cached=0 missing=0 model=dinov2_small@ed25f3a31f01 run_id=b2dc8110-dfd0-4767-9698-84579739de8c
[exit code 0, 4.7 s]


**What this shows.** Each dataset's version, and the real model run once over every photo; the vectors are cached inside each dataset.

## 2. Are there near copies in COCO128 itself?

`split --near-dup` groups every pair of photos whose embeddings have a cosine similarity of at least the threshold (0.95 by default),
all pairs and not only a few nearest neighbours. The cell counts those pairs in the untouched COCO128 at several thresholds, with the
same exact search the split uses, and in the copy with the planted copies.

In [5]:
import uuid

import datacopilot as dc
from datacopilot.audit.context import CheckContext
from datacopilot.audit.embedding import PipelineProvider
from datacopilot.audit.usecase import resolve_embedder
from datacopilot.embed import knn
from datacopilot.logging import configure_logging
from datacopilot.settings import load_settings

settings = load_settings()
configure_logging(
    "datacopilot", settings
)  # JSON records on standard error, errors only (set in the first cell)
embedder = resolve_embedder("dinov2_small", settings.allowed_licenses)
root = Path.cwd().resolve()


def image_embeddings(path):
    dataset = dc.load(path)
    provider = PipelineProvider(embedder, use_cache=True)
    ctx = CheckContext(
        run_id=str(uuid.uuid4()),
        root=root,
        cache_dir=settings.cache_dir,
        dataset_dir=root / path,
        cache_key=settings.cache_key,
        embeddings=provider,
        embedder="dinov2_small",
    )
    return dataset, provider, ctx, provider.get(dataset, ctx)


def pair_count(vectors, threshold):
    found = []
    knn.pairs_at_least(
        vectors,
        threshold,
        lambda a, b: found.extend(zip(a.tolist(), b.tolist(), strict=True)),
        max_pairs=1_000_000,
    )
    return found


clean, provider, first_ctx, embeddings = image_embeddings("coco128.dcp")
copies, _, _, copy_embeddings = image_embeddings("copies.dcp")
print(
    "cache state of the cached vectors:",
    embeddings.cache_state,
    "| spot-checked:",
    embeddings.spot_checked,
    "| mismatched:",
    embeddings.spot_mismatched,
)
for threshold in (0.95, 0.9, 0.85, 0.8):
    print(
        f"COCO128, pairs at cosine >= {threshold}: {len(pair_count(embeddings.matrix, threshold))}"
    )
copy_pairs = pair_count(copy_embeddings.matrix, 0.95)
name_of_id = {i.image_id: Path(i.file_path).name for i in copies.images}
sims = knn.unit_rows(copy_embeddings.matrix)
print(f"with the planted copies, pairs at cosine >= 0.95: {len(copy_pairs)}")
for a, b in copy_pairs:
    print(
        f"  {name_of_id[copy_embeddings.ids[a]]} ~ {name_of_id[copy_embeddings.ids[b]]}: cosine {float(sims[a] @ sims[b]):.4f}"
    )

cache state of the cached vectors: ok | spot-checked: 10 | mismatched: 0
COCO128, pairs at cosine >= 0.95: 0
COCO128, pairs at cosine >= 0.9: 0
COCO128, pairs at cosine >= 0.85: 0
COCO128, pairs at cosine >= 0.8: 0
with the planted copies, pairs at cosine >= 0.95: 5
  copy_000000000605.jpg ~ 000000000605.jpg: cosine 0.9950
  copy_000000000404.jpg ~ 000000000404.jpg: cosine 0.9947
  copy_000000000474.jpg ~ 000000000474.jpg: cosine 0.9937
  000000000431.jpg ~ copy_000000000431.jpg: cosine 0.9960
  000000000595.jpg ~ copy_000000000595.jpg: cosine 0.9961


**What this shows.** How many near-copy pairs the untouched sample holds at each threshold, and that every planted copy is
found next to its original. If COCO128 holds none, a near-duplicate split of it changes nothing that a plain split would not; the
planted copies are what the next cells test.

## 3. The leak, before the split

The leakage check searches every split for its nearest photos in the *other* splits. With `--fail-on leakage` a found leak is exit code 1.

In [6]:
def leakage_audit(path, name):
    done = datacopilot(
        "audit",
        path,
        "--checks",
        "leakage,near_duplicates",
        "--embedder",
        "dinov2_small",
        "--no-hash-cache",
        "--fail-on",
        "leakage",
        "--json",
        f"{name}.json",
        "--report",
        f"{name}.html",
    )
    found = json.loads(Path(f"{name}.json").read_text())["findings"]
    by_rule = Counter((f["check"], f["payload"]["rule"]) for f in found)
    print("findings by check and rule:", dict(sorted(by_rule.items())))
    return done.returncode, [
        f for f in found if f["payload"]["rule"] == "cross_split_near_duplicate"
    ]


before_code, before_leaks = leakage_audit("copies.dcp", "before")
print("cross-split clusters before the split:", len(before_leaks), "| exit code:", before_code)

$ datacopilot audit copies.dcp --checks leakage,near_duplicates --embedder dinov2_small --no-hash-cache --fail-on leakage --json before.json --report before.html
failed critical=10 warn=0 info=0 run_id=55515745-9301-4f49-b639-dda87b5402d0
[exit code 1]
findings by check and rule: {('leakage', 'cross_split_near_duplicate'): 5, ('near_duplicates', 'near_duplicate'): 5}
cross-split clusters before the split: 5 | exit code: 1


**What this shows.** The leakage gate on the dataset with the planted copies: the clusters it reports across splits, and the exit code CI would see.

## 4. `split --near-dup`, then the same check

`split` writes a **new** dataset (the original is not touched; the new one names it in its lineage). Groups are photos that share a
`group_id` (none here) and every near-copy pair; whole groups are placed, largest first, in the split that keeps the photo counts and,
with `--stratify category`, the box counts per class closest to the ratios. The report says how far each split and class ended from
its target.

In [7]:
split_run = datacopilot(
    "split",
    "copies.dcp",
    "-o",
    "split.dcp",
    "--ratios",
    "0.8,0.1,0.1",
    "--near-dup",
    "--embedder",
    "dinov2_small",
    "--stratify",
    "category",
    "--seed",
    "0",
    "--json",
    "split.json",
    timed=True,
)
report = json.loads(Path("split.json").read_text())
for key in (
    "image_count",
    "group_count",
    "multi_image_groups",
    "largest_group",
    "near_duplicate_pairs",
    "groups_across_splits",
    "moved_images",
    "max_class_deviation",
):
    print(f"{key:<22}: {report[key]}")
worst = sorted(
    (
        (abs(c["deviation"]), s, cid, c)
        for s, row in report["classes"].items()
        for cid, c in row.items()
    ),
    reverse=True,
)[:3]
for _, split_name, cid, c in worst:
    print(
        f"largest class deviations: {split_name} class {cid}: {c['annotations']} boxes, share {c['share']}, deviation {c['deviation']:+}"
    )

$ datacopilot split copies.dcp -o split.dcp --ratios 0.8,0.1,0.1 --near-dup --embedder dinov2_small --stratify category --seed 0 --json split.json
198c79effcd84052658e660340ffd864
images=133 groups=128 moved=46 near_duplicate_pairs=5 groups_across_splits=0 max_class_deviation=0.2
train images=109 target=106.4 deviation=+0.0195
val images=12 target=13.3 deviation=-0.0098
test images=12 target=13.3 deviation=-0.0098
[exit code 0, 2.9 s]
image_count           : 133
group_count           : 128
multi_image_groups    : 5
largest_group         : 2
near_duplicate_pairs  : 5
groups_across_splits  : 0
moved_images          : 46
max_class_deviation   : 0.2
largest class deviations: train class 79: 5 boxes, share 1.0, deviation +0.2
largest class deviations: train class 77: 21 boxes, share 1.0, deviation +0.2
largest class deviations: train class 76: 1 boxes, share 1.0, deviation +0.2


**What this shows.** The new version, the groups the split built, and the deviation from the ratios per split and for the classes that are furthest off.

In [8]:
after = dc.load("split.dcp")
split_of = {Path(i.file_path).name: i.split for i in after.images}
together = [split_of[o] == split_of[c] for o, c in planted_copies.items()]
print("planted copies on the same side as their original:", sum(together), "of", len(together))
print(
    "lineage of the new version ends with the old one:",
    after.manifest.lineage[-1] == copies.manifest.version,
)
print(
    "the old dataset still has its version:",
    dc.load("copies.dcp").manifest.version == copies.manifest.version,
)
datacopilot("embed", "split.dcp", "--embedder", "dinov2_small")
after_code, after_leaks = leakage_audit("split.dcp", "after")
print("cross-split clusters after the split:", len(after_leaks), "| exit code:", after_code)

planted copies on the same side as their original: 5 of 5
lineage of the new version ends with the old one: True
the old dataset still has its version: True


$ datacopilot embed split.dcp --embedder dinov2_small
embedded=133 total=133 cached=0 missing=0 model=dinov2_small@ed25f3a31f01 run_id=b55a0e09-1967-4340-9537-37e227c62473
[exit code 0]


$ datacopilot audit split.dcp --checks leakage,near_duplicates --embedder dinov2_small --no-hash-cache --fail-on leakage --json after.json --report after.html
passed critical=0 warn=5 info=0 run_id=3434ad3c-f01e-4b94-984d-a02dcf41d59c
[exit code 0]
findings by check and rule: {('near_duplicates', 'near_duplicate'): 5}
cross-split clusters after the split: 0 | exit code: 0


**What this shows.** Whether each copy landed with its original, that the old version is unchanged, and the same leakage gate on the new version.

In [9]:
again = datacopilot(
    "split",
    "copies.dcp",
    "-o",
    "split_again.dcp",
    "--ratios",
    "0.8,0.1,0.1",
    "--near-dup",
    "--embedder",
    "dinov2_small",
    "--stratify",
    "category",
    "--seed",
    "0",
)
print(
    "the same input, ratios and seed give the same version:",
    again.stdout.splitlines()[0] == split_run.stdout.splitlines()[0],
)
other_seed = datacopilot(
    "split",
    "copies.dcp",
    "-o",
    "split_seed1.dcp",
    "--ratios",
    "0.8,0.1,0.1",
    "--near-dup",
    "--embedder",
    "dinov2_small",
    "--stratify",
    "category",
    "--seed",
    "1",
)
moved = sum(
    1 for i in dc.load("split_seed1.dcp").images if split_of[Path(i.file_path).name] != i.split
)
print("another seed moves", moved, "photos")

$ datacopilot split copies.dcp -o split_again.dcp --ratios 0.8,0.1,0.1 --near-dup --embedder dinov2_small --stratify category --seed 0
198c79effcd84052658e660340ffd864
images=133 groups=128 moved=46 near_duplicate_pairs=5 groups_across_splits=0 max_class_deviation=0.2
train images=109 target=106.4 deviation=+0.0195
val images=12 target=13.3 deviation=-0.0098
test images=12 target=13.3 deviation=-0.0098
[exit code 0]
the same input, ratios and seed give the same version: True


$ datacopilot split copies.dcp -o split_seed1.dcp --ratios 0.8,0.1,0.1 --near-dup --embedder dinov2_small --stratify category --seed 1
ba1f4f00f7ffb31b09885fc14e920a41
images=133 groups=128 moved=45 near_duplicate_pairs=5 groups_across_splits=0 max_class_deviation=0.2
train images=108 target=106.4 deviation=+0.0120
val images=13 target=13.3 deviation=-0.0023
test images=12 target=13.3 deviation=-0.0098
[exit code 0]
another seed moves 13 photos


**What this shows.** The split is reproducible, and the seed is the only thing besides the data and the ratios that changes it.

## 5. Label noise: box crops against whole photos

The protocol of notebook 04, unchanged: a seeded share of the boxes (`RATE`) gets a class chosen uniformly among the other classes
of the dataset, five seeds, caps lifted for the measurement. A box is **found** when the check reports it. Every measure has two
denominators: all boxes, and the boxes the rule could score.

- **image rule** (`suspect_basis="image"`): notebook 04's rule, re-measured here on the same photos and the same seeds.
- **crop rule** (the default now): each box of at least `crop_min_side_px` pixels (shorter side) is cut with `crop_padding` of
  context, at most `max_crops_per_image` per image, and embedded; a box is suspect when its nearest crops do not carry its class.

In [10]:
from datacopilot.audit import engine
from datacopilot.audit.context import CheckConfig
from datacopilot.audit.crops import CropPlan
from datacopilot.core.schema import Dataset

DEFAULT = CheckConfig()
SEEDS = (0, 1, 2, 3, 4)
RATE = 0.05
UNCAPPED = dict(max_suspects_per_image=1000, max_suspects_per_class=10_000)
print("planted share (RATE):", RATE, "| seeds:", SEEDS)
print(
    "crop settings: min side",
    DEFAULT.crop_min_side_px,
    "px, padding",
    DEFAULT.crop_padding,
    ", at most",
    DEFAULT.max_crops_per_image,
    "per image",
)
crop_sets, crop_seconds = {}, {}


def crops_for(min_side=DEFAULT.crop_min_side_px):
    if min_side not in crop_sets:
        t0 = time.perf_counter()
        crop_sets[min_side] = provider.crops(
            clean, first_ctx, CropPlan(min_side, DEFAULT.crop_padding, DEFAULT.max_crops_per_image)
        )
        crop_seconds[min_side] = time.perf_counter() - t0
    return crop_sets[min_side]


cs = crops_for()
print(
    f"boxes: {cs.boxes}; cropped and embedded: {len(cs.es.ids)} crops ({cs.es.total} planned); not cropped: {cs.skipped}; "
    f"not embedded: { {k: len(v) for k, v in cs.es.missing.items()} }; {crop_seconds[DEFAULT.crop_min_side_px]:.1f} s with the real model"
)
print("crop cache: spot-checked", cs.es.spot_checked, "| mismatched", cs.es.spot_mismatched)

planted share (RATE): 0.05 | seeds: (0, 1, 2, 3, 4)
crop settings: min side 32 px, padding 0.1 , at most 50 per image


boxes: 929; cropped and embedded: 513 crops (513 planned); not cropped: {'too_small': 416}; not embedded: {}; 8.5 s with the real model
crop cache: spot-checked 0 | mismatched 0


**What this shows.** How many boxes are large enough to crop at the default size floor, and the time the model took for them.

In [11]:
def plant(dataset, seed, mode="random", rate=RATE):
    """Copy of the dataset with a seeded share of boxes given a wrong class; returns it, {ann_id: (old, new)} and the eligible count.

    mode "same_photo": only boxes of photos that hold another class are eligible, and the new class is one of
    that photo's other classes (a confusion with something that really is there); there is no fallback.
    """
    rng = np.random.default_rng(seed)
    anns = list(dataset.annotations)
    present = sorted({a.category_id for a in anns})
    in_photo = {}
    for a in anns:
        in_photo.setdefault(a.image_id, set()).add(a.category_id)
    eligible = [i for i, a in enumerate(anns) if mode == "random" or len(in_photo[a.image_id]) > 1]
    chosen = sorted(rng.choice(eligible, round(rate * len(anns)), replace=False))
    out, planted = list(anns), {}
    for i in chosen:
        a = anns[i]
        pool = [c for c in present if c != a.category_id]
        if mode == "same_photo":
            pool = sorted(in_photo[a.image_id] - {a.category_id})
        new = int(rng.choice(pool))
        out[i] = a.model_copy(update={"category_id": new})
        planted[a.ann_id] = (a.category_id, new)
    return (
        Dataset(dataset.manifest, dataset.images, tuple(out), dataset.root),
        planted,
        len(eligible),
    )


class Both:
    """The verified image vectors and crop vectors computed above (the photos never change, only the labels)."""

    def __init__(self, min_side=DEFAULT.crop_min_side_px):
        self.min_side = min_side

    def get(self, dataset, ctx):
        return embeddings

    def crops(self, dataset, ctx, plan):
        return crops_for(self.min_side)


def run_check(dataset, min_side=DEFAULT.crop_min_side_px, **config):
    ctx = CheckContext(
        run_id=str(uuid.uuid4()),
        root=root,
        cache_dir=settings.cache_dir,
        embeddings=Both(min_side),
        embedder="dinov2_small",
        config=CheckConfig(crop_min_side_px=min_side, **config),
    )
    return engine.run_audit(dataset, ["label_noise"], ctx)


def suspects_of(run):
    return {f.ann_id: f for f in run.findings if f.payload["rule"] == "suspect_class"}


def unscored_ids(
    dataset,
    basis,
    min_side=DEFAULT.crop_min_side_px,
    suspect_k=DEFAULT.suspect_k,
    suspect_min_similarity=DEFAULT.suspect_min_similarity,
    suspect_min_neighbours=DEFAULT.suspect_min_neighbours,
    suspect_max_support=DEFAULT.suspect_max_support,
):
    """Independent re-derivation in plain NumPy: (suspect ann ids, unscored ann ids) of either rule."""
    if basis == "image":
        ids = list(embeddings.ids)
        rows = [i for i in ids if any(a.image_id == i for a in dataset.annotations)]
        members = {i: [a for a in dataset.annotations if a.image_id == i] for i in rows}
        matrix = embeddings.matrix[[ids.index(i) for i in rows]]
        unscored = set()
    else:
        c = crops_for(min_side)
        rows = list(c.es.ids)
        members = {key: [] for key in rows}
        for a in dataset.annotations:
            key = c.key_of.get(a.ann_id)
            if key in members:
                members[key].append(a)
        matrix = c.es.matrix
        unscored = {a.ann_id for a in dataset.annotations if c.key_of.get(a.ann_id) not in members}
    m = matrix.astype(np.float64)
    m /= np.linalg.norm(m, axis=1, keepdims=True)
    sim = m @ m.T
    np.fill_diagonal(sim, -np.inf)
    carried = [{a.category_id for a in members[r]} for r in rows]
    suspects = set()
    for n, r in enumerate(rows):
        order = np.lexsort((np.arange(len(rows)), -sim[n]))[: min(suspect_k, len(rows) - 1)]
        usable = [j for j in order if sim[n, j] >= suspect_min_similarity - 1e-6]
        if len(usable) < suspect_min_neighbours:
            unscored.update(a.ann_id for a in members[r])
            continue
        for a in members[r]:
            if (
                sum(a.category_id in carried[j] for j in usable) / len(usable)
                <= suspect_max_support
            ):
                suspects.add(a.ann_id)
    return suspects, unscored


def measure(dataset, planted, basis, min_side=DEFAULT.crop_min_side_px, **config):
    run = run_check(dataset, min_side=min_side, suspect_basis=basis, **config)
    reported = suspects_of(run)
    mine, unscored = unscored_ids(
        dataset,
        basis,
        min_side=min_side,
        **{k: v for k, v in config.items() if k.startswith("suspect_")},
    )
    total = len(dataset.annotations)
    scored = total - len(unscored)
    found = len(set(reported) & set(planted))
    planted_scorable = len(set(planted) - unscored)
    false_alarms = len(reported) - found
    return dict(
        planted=len(planted),
        reported=len(reported),
        found=found,
        scored=scored,
        planted_scorable=planted_scorable,
        precision=found / len(reported) if reported else float("nan"),
        recall=found / len(planted) if planted else float("nan"),
        recall_scorable=found / planted_scorable if planted_scorable else float("nan"),
        fpr=false_alarms / (total - len(planted)),
        fpr_scorable=false_alarms / (scored - planted_scorable)
        if scored > planted_scorable
        else float("nan"),
        agrees=mine == set(reported),
        check_ms=run.timings_ms["label_noise"],
        run=run,
    )


results = {"image": {}, "crop": {}}
planted_sets = {}
for seed in SEEDS:
    damaged, planted, _ = plant(clean, seed)
    planted_sets[seed] = (damaged, planted)
    for basis in results:
        results[basis][seed] = measure(damaged, planted, basis, **UNCAPPED)


def mean(key, basis, rows=None):
    rows = rows or [results[basis][s] for s in SEEDS]
    return float(np.mean([r[key] for r in rows]))


for basis in ("image", "crop"):
    print(f"{basis.upper()} RULE, over all boxes")
    print(
        f"{'seed':>4} {'planted':>8} {'reported':>9} {'found':>6} {'precision':>10} {'recall':>8} {'FP rate':>8} {'scored':>7}"
    )
    for seed in SEEDS:
        r = results[basis][seed]
        print(
            f"{seed:>4} {r['planted']:>8} {r['reported']:>9} {r['found']:>6} {r['precision']:>10.3f} {r['recall']:>8.3f} {r['fpr']:>8.3f} {r['scored']:>7}"
        )
    print(
        f"{'mean':>4} {mean('planted', basis):>8.1f} {mean('reported', basis):>9.1f} {mean('found', basis):>6.1f} {mean('precision', basis):>10.3f} {mean('recall', basis):>8.3f} {mean('fpr', basis):>8.3f} {mean('scored', basis):>7.1f}"
    )
    print()
agreement = all(r["agrees"] for b in results for r in results[b].values())
print(
    "the check and the independent re-derivation report the same boxes in all 10 runs:", agreement
)
if not agreement:
    raise RuntimeError("the check and the independent version disagree")

IMAGE RULE, over all boxes
seed  planted  reported  found  precision   recall  FP rate  scored
   0       46        97     14      0.144    0.304    0.094     296
   1       46        93      7      0.075    0.152    0.097     296
   2       46        96     10      0.104    0.217    0.097     296
   3       46        97     13      0.134    0.283    0.095     296
   4       46        99      8      0.081    0.174    0.103     296
mean     46.0      96.4   10.4      0.108    0.226    0.097   296.0

CROP RULE, over all boxes
seed  planted  reported  found  precision   recall  FP rate  scored
   0       46        94     27      0.287    0.587    0.076     455
   1       46        83     22      0.265    0.478    0.069     455
   2       46        81     17      0.210    0.370    0.072     455
   3       46        88     23      0.261    0.500    0.074     455
   4       46        84     19      0.226    0.413    0.074     455
mean     46.0      86.0   21.6      0.250    0.470    0.073   

**What this shows.** Both rules on the same planted mistakes, seed by seed, with how many boxes each could score; and that a separate few-line version of each rule reports exactly the same boxes.

In [12]:
import re

PHASE6 = None
record = Path(dc.__file__).resolve().parents[2] / "notebooks" / "04_label_noise.ipynb"
if record.is_file():  # the saved notebook 04, if this is a clone of the repository
    saved = "".join(
        "".join(o.get("text", ""))
        for c in json.loads(record.read_text())["cells"]
        if c["cell_type"] == "code"
        for o in c["outputs"]
    )
    row = re.search(
        r"OVER ALL BOXES.*?\nmean\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)",
        saved,
        re.S,
    )
    if row:
        PHASE6 = dict(
            zip(
                ("planted", "reported", "found", "precision", "recall", "fpr", "not_scored"),
                map(float, row.groups()),
                strict=True,
            )
        )
if PHASE6:
    same = all(
        abs(PHASE6[k] - round(mean(k, "image"), 3)) < 1e-9 for k in ("precision", "recall", "fpr")
    )
    print(
        "notebook 04 as saved (Phase 6, image rule):",
        {k: PHASE6[k] for k in ("precision", "recall", "fpr")},
        "| boxes scored:",
        round(len(clean.annotations) - PHASE6["not_scored"]),
    )
    print("the image rule re-measured here gives the same three numbers:", same)
else:
    print(
        "notebook 04 is not next to this installation; the image rule re-measured above stands in for it"
    )
print()
print(f"{'':<30} {'image rule':>11} {'crop rule':>10}")
for label, key in (
    ("precision", "precision"),
    ("recall, all boxes", "recall"),
    ("FP rate, all boxes", "fpr"),
    ("recall, scorable boxes", "recall_scorable"),
    ("FP rate, scorable boxes", "fpr_scorable"),
):
    print(f"{label:<30} {mean(key, 'image'):>11.3f} {mean(key, 'crop'):>10.3f}")
print(
    f"{'boxes scored (of ' + str(len(clean.annotations)) + ')':<30} {mean('scored', 'image'):>11.0f} {mean('scored', 'crop'):>10.0f}"
)
print(
    f"{'reports per run':<30} {mean('reported', 'image'):>11.1f} {mean('reported', 'crop'):>10.1f}"
)
untouched = {b: measure(clean, {}, b, **UNCAPPED) for b in ("image", "crop")}
print(
    f"{'untouched data: FP rate, all':<30} {untouched['image']['fpr']:>11.3f} {untouched['crop']['fpr']:>10.3f}"
)

notebook 04 as saved (Phase 6, image rule): {'precision': 0.108, 'recall': 0.226, 'fpr': 0.097} | boxes scored: 296
the image rule re-measured here gives the same three numbers: True

                                image rule  crop rule
precision                            0.108      0.250
recall, all boxes                    0.226      0.470
FP rate, all boxes                   0.097      0.073
recall, scorable boxes               0.831      0.962
FP rate, scorable boxes              0.303      0.149
boxes scored (of 929)                  296        455
reports per run                       96.4       86.0
untouched data: FP rate, all         0.097      0.071


**What this shows.** The comparison the change was made for, every value a mean over the five seeds; the first lines compare the
image rule measured here with the numbers notebook 04 saved. The untouched dataset may hold real mistakes, so its reports are not all false alarms.

In [13]:
same_photo = {}
for basis in ("image", "crop"):
    rows = []
    for seed in SEEDS:
        damaged, planted, eligible = plant(clean, seed, mode="same_photo")
        rows.append(measure(damaged, planted, basis, **UNCAPPED))
    same_photo[basis] = rows
    print(
        f"{basis} rule, confusion with another class of the same photo: precision {mean('precision', basis, rows):.3f}, "
        f"recall {mean('recall', basis, rows):.3f} (all boxes), {mean('recall_scorable', basis, rows):.3f} (scorable boxes)"
    )

image rule, confusion with another class of the same photo: precision 0.047, recall 0.091 (all boxes), 0.276 (scorable boxes)


crop rule, confusion with another class of the same photo: precision 0.156, recall 0.252 (all boxes), 0.539 (scorable boxes)


**What this shows.** The harder case of notebook 04: a box given a class that really is in the same photo. A whole-photo vector cannot tell those apart; a crop can try.

In [14]:
def write_damaged_yolo(planted, target):
    """Copy the YOLO folder and rewrite the class number of every planted box (ann ids are '<split>/<photo>.jpg:<line>')."""
    shutil.copytree("coco128", target)
    by_file = {}
    for ann_id, (old, new) in planted.items():
        photo, line = ann_id.rsplit(":", 1)
        split, name = photo.split("/")
        by_file.setdefault(f"labels/{split}/{Path(name).stem}.txt", {})[int(line)] = (old, new)
    touched = 0
    for label_file, edits in by_file.items():
        path = Path(target) / label_file
        lines = path.read_text().splitlines()
        for n, (old, new) in edits.items():
            parts = lines[n].split()
            if int(parts[0]) != old:
                raise RuntimeError("label line does not hold the expected class")
            lines[n] = " ".join([str(new), *parts[1:]])
            touched += 1
        path.write_text("\n".join(lines) + "\n")
    return touched


damaged0, planted0 = planted_sets[0]
print("label lines rewritten:", write_damaged_yolo(planted0, "damaged0"))
datacopilot("ingest", "damaged0", "--format", "yolo", "-o", "damaged0.dcp")
datacopilot("embed", "damaged0.dcp", "--embedder", "dinov2_small")
datacopilot(
    "audit",
    "damaged0.dcp",
    "--checks",
    "label_noise",
    "--embedder",
    "dinov2_small",
    "--no-hash-cache",
    "--fail-on",
    "none",
    "--json",
    "damaged0.json",
    timed=True,
)
cli_report = json.loads(Path("damaged0.json").read_text())
cli_ids = {f["ann_id"] for f in cli_report["findings"] if f["payload"]["rule"] == "suspect_class"}
bases = {
    f["payload"].get("basis")
    for f in cli_report["findings"]
    if f["payload"]["rule"] == "suspect_class"
}
api_ids = set(suspects_of(run_check(damaged0)))
print(
    "command line: rule basis",
    bases,
    "|",
    len(cli_ids),
    "boxes; in-process with the same (default) caps:",
    len(api_ids),
)
print("same boxes:", cli_ids == api_ids)
for f in cli_report["findings"]:
    if f["payload"]["rule"] in ("annotations_not_scored", "crop_rule_unavailable"):
        print(f["severity"], f["payload"])
if cli_ids != api_ids:
    raise RuntimeError("the command line and the in-process run disagree")

label lines rewritten: 46


$ datacopilot ingest damaged0 --format yolo -o damaged0.dcp
8158718709735f81054da764d536cdd6
[exit code 0]


$ datacopilot embed damaged0.dcp --embedder dinov2_small
embedded=128 total=128 cached=0 missing=0 model=dinov2_small@ed25f3a31f01 run_id=5ca8a6ac-f02c-4340-80ad-410003301433
[exit code 0]


$ datacopilot audit damaged0.dcp --checks label_noise --embedder dinov2_small --no-hash-cache --fail-on none --json damaged0.json
passed critical=1 warn=0 info=94 run_id=c3bab2b8-aa44-4e5b-9068-97c5e709e5cc
[exit code 0, 21.2 s]


command line: rule basis {'crop'} | 94 boxes; in-process with the same (default) caps: 94
same boxes: True
CRITICAL {'count': 474, 'min_neighbours': 3, 'min_similarity': 0.3, 'reasons': {'too_few_neighbours': 58, 'too_small': 416}, 'rule': 'annotations_not_scored', 'total': 929}


**What this shows.** The command line, on a dataset whose label files really carry seed 0's planted classes, uses the crop rule by default and reports the same boxes as the measurement; the count of boxes it could not score is part of the report, by reason.

## 6. Other settings (tuned on this same data, so an upper bound)

The defaults were fixed before this notebook was run. The grid changes the crop size floor, the similarity floor and the support
threshold afterwards, looking at the answer key, so it flatters the rule.

In [15]:
print(
    f"{'min side':>8} {'sim floor':>9} {'max support':>11} {'reported':>9} {'precision':>10} {'recall':>8} {'FP rate':>8} {'scored':>7}"
)
grid = {}
for min_side in (16, 32):
    for floor in (0.3, 0.5):
        for support in (0.0, 0.2):
            cells = [
                measure(
                    *planted_sets[s],
                    "crop",
                    min_side=min_side,
                    suspect_min_similarity=floor,
                    suspect_max_support=support,
                    **UNCAPPED,
                )
                for s in SEEDS
            ]
            g = grid[(min_side, floor, support)] = {
                k: float(np.mean([c[k] for c in cells]))
                for k in ("reported", "precision", "recall", "fpr", "scored")
            }
            print(
                f"{min_side:>8} {floor:>9} {support:>11} {g['reported']:>9.1f} {g['precision']:>10.3f} {g['recall']:>8.3f} {g['fpr']:>8.3f} {g['scored']:>7.0f}"
            )
best_p = max(grid.items(), key=lambda kv: kv[1]["precision"])
best_r = max(grid.items(), key=lambda kv: kv[1]["recall"])
print(
    f"highest precision in the grid: {best_p[1]['precision']:.3f} (min side {best_p[0][0]}, floor {best_p[0][1]}, support {best_p[0][2]}) at recall {best_p[1]['recall']:.3f}"
)
print(
    f"highest recall in the grid: {best_r[1]['recall']:.3f} (min side {best_r[0][0]}, floor {best_r[0][1]}, support {best_r[0][2]}) at precision {best_r[1]['precision']:.3f}"
)

min side sim floor max support  reported  precision   recall  FP rate  scored


      16       0.3         0.0     125.8      0.243    0.665    0.108     659


      16       0.3         0.2     232.8      0.137    0.696    0.227     659


      16       0.5         0.0      68.6      0.248    0.370    0.058     363


      16       0.5         0.2      97.8      0.178    0.378    0.091     363


      32       0.3         0.0      86.0      0.250    0.470    0.073     455


      32       0.3         0.2     142.6      0.155    0.483    0.136     455


      32       0.5         0.0      31.2      0.318    0.217    0.024     192


      32       0.5         0.2      36.6      0.277    0.222    0.030     192
highest precision in the grid: 0.318 (min side 32, floor 0.5, support 0.0) at recall 0.217
highest recall in the grid: 0.696 (min side 16, floor 0.3, support 0.2) at precision 0.137


**What this shows.** What each setting costs; read precision and recall together.

## 7. Runtime

In [16]:
check_ms = {b: [results[b][s]["check_ms"] for s in SEEDS] for b in ("image", "crop")}
print(
    f"embedding {len(clean.images)} photos (command line, including start-up): {EMBED_SECONDS:.1f} s"
)
print(
    f"cutting and embedding {len(cs.es.ids)} crops (in process, real model): {crop_seconds[DEFAULT.crop_min_side_px]:.1f} s"
)
for b in ("image", "crop"):
    print(
        f"label_noise itself, {b} rule, per run: min {min(check_ms[b])} ms, max {max(check_ms[b])} ms (cached vectors)"
    )

embedding 128 photos (command line, including start-up): 5.5 s
cutting and embedding 513 crops (in process, real model): 8.5 s
label_noise itself, image rule, per run: min 7 ms, max 7 ms (cached vectors)
label_noise itself, crop rule, per run: min 11 ms, max 12 ms (cached vectors)


**What this shows.** The crop rule's cost is the model running once per box instead of once per photo; the check itself stays a search over cached vectors.

## 8. Verdict, computed from the numbers above

In [17]:
from IPython.display import Markdown, display

p, r, f = mean("precision", "crop"), mean("recall", "crop"), mean("fpr", "crop")
pi, ri, fi = mean("precision", "image"), mean("recall", "image"), mean("fpr", "image")
lines = [
    f"- Split: COCO128 holds {len(pair_count(embeddings.matrix, 0.95))} near-copy pairs at cosine 0.95. With {len(planted_copies)} planted copies the leakage gate reported {len(before_leaks)} cross-split clusters (exit {before_code}); after `split --near-dup` it reported {len(after_leaks)} (exit {after_code}), and {sum(together)} of {len(together)} copies sit with their original.",
    f"- Crop rule against the image rule (means of five seeds, caps lifted): precision {p:.2f} against {pi:.2f}, recall {r:.2f} against {ri:.2f}, false-positive rate {f:.2f} against {fi:.2f}, boxes scored {mean('scored', 'crop'):.0f} against {mean('scored', 'image'):.0f} of {len(clean.annotations)}.",
    f"- Same-photo confusions: recall {mean('recall', 'crop', same_photo['crop']):.2f} with crops against {mean('recall', 'image', same_photo['image']):.2f} with whole photos.",
    f"- Boxes below the {DEFAULT.crop_min_side_px}-pixel floor are not cropped: {cs.skipped.get('too_small', 0)} of {cs.boxes} here; they are counted as not scored, never as clean.",
]
better = p > pi and r > ri and f < fi
verdict = (
    "**The crop rule is better than the image rule on every one of the three measures here.** "
    if better
    else "**The crop rule is not better than the image rule on every measure here.** "
)
if p < 0.5 or r < 0.5:
    verdict += (
        f"**It is still weak:** {1 - p:.0%} of what it reports is not a planted mistake, and it finds {r:.0%} of them. "
        "Treat a report as a short list of leads, never as a list of errors, and do not gate on it."
    )
else:
    verdict += (
        "Most of its reports are planted mistakes and it finds most of them, on this dataset."
    )
display(Markdown("\n".join(lines) + "\n\n" + verdict))

- Split: COCO128 holds 0 near-copy pairs at cosine 0.95. With 5 planted copies the leakage gate reported 5 cross-split clusters (exit 1); after `split --near-dup` it reported 0 (exit 0), and 5 of 5 copies sit with their original.
- Crop rule against the image rule (means of five seeds, caps lifted): precision 0.25 against 0.11, recall 0.47 against 0.23, false-positive rate 0.07 against 0.10, boxes scored 455 against 296 of 929.
- Same-photo confusions: recall 0.25 with crops against 0.09 with whole photos.
- Boxes below the 32-pixel floor are not cropped: 416 of 929 here; they are counted as not scored, never as clean.

**The crop rule is better than the image rule on every one of the three measures here.** **It is still weak:** 75% of what it reports is not a planted mistake, and it finds 47% of them. Treat a report as a short list of leads, never as a list of errors, and do not gate on it.

**What this shows.** The wording follows from the numbers and the thresholds in the code, not from a sentence written in advance.

## What you learned

- `split` keeps every near-copy pair and every `group_id` on one side, writes a new version with lineage, and says how far it ended
  from the ratios; with nothing to group it is an ordinary stratified split.
- The leakage check is the independent judge of a split: the cells above ran it before and after on the same planted copies.
- `label_noise` now embeds box crops by default. The tables above compare it with the whole-photo rule on the same planted mistakes;
  boxes too small to crop are counted, not hidden.
- Confident learning (the Cleanlab backend) and class-aware calibration are not built; see the decision record.